# Banc de décisions Deposium

Interface du moteur commun CLI/notebook. Le profil par défaut est un **contrôle sur réponses figées**, pas un modèle. Aucun poids, API ou service externe ne sont appelés par défaut. Le replay MCPs demande un profil et une stack explicites; il appelle une fonction réelle sans qualifier la route HTTP ni le produit. Voir `docs/NOTEBOOK.md`.


In [ ]:
# Paramètres simples; les valeurs dérivées sont calculées après injection.
dataset_path = "datasets/decisions-v1/cases.jsonl"
profile_path = "profiles/fixture-control.json"
output_dir = None
stack_root = None
run_mode = "run"  # run, resume, rescore
source_archive = None
max_new_cases = None


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "pyproject.toml").is_file()
     and (parent / "README.md").is_file()
     and (parent / "deposium_bench").is_dir()),
    None,
)
if root is None:
    raise RuntimeError("Lancer le kernel depuis le clone du banc ou notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from deposium_bench.campaigns import (
    run_campaign, resume_campaign, rescore_campaign, render_report,
)

# Toutes les valeurs dérivées sont résolues après la cellule de paramètres.
def resolve_path(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path if path.is_absolute() else root / path

archive_destination = resolve_path(output_dir) if output_dir is not None else (
    root / "results" / f"notebook-{uuid4().hex}"
)
resolved_dataset = resolve_path(dataset_path)
resolved_profile = resolve_path(profile_path)
resolved_stack = resolve_path(stack_root)
resolved_source = resolve_path(source_archive)


## Exécution et archive

Chaque appel crée une nouvelle archive. Reprise et rescore gardent les preuves initiales. `max_new_cases` permet une campagne partielle; les cas absents restent visibles. Les annotations et scorers restent hors des cellules et hors de l’entrée du candidat.


In [ ]:
if run_mode == "run":
    summary = run_campaign(
        resolved_dataset, resolved_profile, archive_destination,
        stack_root=resolved_stack, max_new_cases=max_new_cases,
    )
elif run_mode in {"resume", "rescore"}:
    if resolved_source is None:
        raise ValueError("source_archive est requis pour resume ou rescore.")
    if run_mode == "resume":
        summary = resume_campaign(
            resolved_source, archive_destination,
            stack_root=resolved_stack, max_new_cases=max_new_cases,
        )
    else:
        summary = rescore_campaign(resolved_source, archive_destination)
else:
    raise ValueError("run_mode doit être run, resume ou rescore.")


In [ ]:
print(render_report(summary))


## Limites de lecture

Afficher couverture, dénominateurs et états avec les scores. Un contrôle figé ne mesure aucun modèle; un replay de fonction MCPs ne mesure pas l’arbitrage intégré ou le parcours produit. Les résultats des modèles et services demandent une campagne distincte avec versions, runtime et prérequis déclarés.

Pour la recherche domaine/longueur, choisir `datasets/context-choice-v1/cases.jsonl` et `profiles/julia-1-cpu-context.json`, ou le JSONL produit par `python3 -m deposium_bench build-lengths`. Le même moteur archive et score ces cas. Le rapport détaillé se construit sans inférence avec `python3 -m deposium_bench analyze-context --archive ... --output ...`. Voir `docs/CONTEXT_BENCH.md`.
